In [13]:
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from io import StringIO
from datetime import datetime

## Parsing the output from Jiggle into a csv 

In [18]:
# Read file, skip the separator line
with open(file_name, 'r') as f:
    lines = f.readlines()

# Get header from first line
header_line = lines[0]
# Skip the separator line (line 1), start data from line 2
data_lines = lines[2:]

# Combine header with data
content = header_line + ''.join(data_lines)

df = pd.read_csv(StringIO(content),
                 sep='\s*\|\s*',
                 engine='python')

# Clean column names
df.columns = df.columns.str.strip()

# Convert data types
df['evid'] = df['evid'].astype(int)
df['etype'] = df['etype'].str.strip()
df['selectflag'] = df['selectflag'].astype(int)

# Fix for inconsistent timestamp format - use format='mixed'
df['to_timestamp'] = pd.to_datetime(df['to_timestamp'], utc = True, format = 'mixed')

df['lat'] = df['lat'].astype(float)
df['lon'] = df['lon'].astype(float)
df['totalarr'] = df['totalarr'].astype(int)

print(df.head())
print(df.info())

       evid etype  selectflag                     to_timestamp        lat  \
0  62215827    su           1 2026-01-24 18:17:17.059998+00:00  43.777500   
1  62223816    su           1 2026-01-21 17:51:48.680000+00:00  43.729500   
2  62223741    su           1 2026-01-21 10:20:43.799999+00:00  48.768667   
3  62223446    su           1 2026-01-19 15:05:39.599999+00:00  48.756500   
4  62223331    su           1 2026-01-18 18:29:43.240000+00:00  43.724667   

          lon  totalarr  
0 -121.259333         5  
1 -121.207333         6  
2 -121.842000         7  
3 -121.842333         5  
4 -121.225167         7  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   evid          346 non-null    int64              
 1   etype         346 non-null    object             
 2   selectflag    346 non-null    int64              
 3

## Filtering out for december 1-17 events

In [31]:
# Just compare dates, not times - this ignores timezone entirely
december_events_pnsn = df[
    (df['to_timestamp'].dt.date >= datetime(2025, 12, 1).date()) & 
    (df['to_timestamp'].dt.date <= datetime(2025, 12, 17).date())
]

print(f"Found {len(december_events_pnsn)} events")
print(december_events_pnsn)

Found 4 events
        evid etype  selectflag                     to_timestamp        lat  \
10  62210017    su           1 2025-12-17 00:45:41.469999+00:00  48.756167   
11  62209537    su           1 2025-12-13 12:54:49.519998+00:00  48.774500   
12  62217776    su           1 2025-12-10 19:57:31.620000+00:00  48.757667   
13  62216746    su           1 2025-12-05 02:03:34.700000+00:00  46.195000   

           lon  totalarr  
10 -121.826000        10  
11 -121.853167         7  
12 -121.847167         7  
13 -122.242500         6  


In [32]:
december_events_pnsn

,evid,etype,selectflag,to_timestamp,lat,lon,totalarr
10,62210017,su,1,2025-12-17 00:45:41.469999+00:00,48.756167,-121.826000,10
11,62209537,su,1,2025-12-13 12:54:49.519998+00:00,48.774500,-121.853167,7
12,62217776,su,1,2025-12-10 19:57:31.620000+00:00,48.757667,-121.847167,7
13,62216746,su,1,2025-12-05 02:03:34.700000+00:00,46.195000,-122.242500,6


In [33]:
common_events = pd.read_csv('../logs/common_2025-12-01T00:00:00_to_2025-12-17T14:20:00_events.csv')

# Fix for inconsistent timestamp format - use format='mixed'
common_events['to_timestamp'] = pd.to_datetime(common_events['rounded_start'], utc = True, format = 'mixed')
common_events

,cluster_id,rounded_start,num_stations,stations,all_classes,most_common_class,mean_auc,mean_max,mean_prob,to_timestamp
0,1,2025-12-01 00:00:49.998571+00:00,5,"['JCW', 'LTY', 'MANO', 'RCM', 'RCS']","['eq', 'eq', 'su', 'su', 'px']",eq,14.295711,0.768417,0.452266,2025-12-01 00:00:49.998571+00:00
1,10,2025-12-01 00:09:00+00:00,4,"['MBW2', 'MILD', 'RER', 'SHUK']","['eq', 'su', 'su', 'eq']",eq,3.844572,0.714808,0.418800,2025-12-01 00:09:00+00:00
2,15,2025-12-01 00:14:50+00:00,4,"['BHW', 'LTY', 'PARA', 'PASS']","['su', 'su', 'eq', 'eq']",eq,4.556851,0.684821,0.408861,2025-12-01 00:14:50+00:00
3,21,2025-12-01 00:22:10+00:00,8,"['BHW', 'GRWR', 'JCW', 'KAVK', 'LTY', 'RCM', '...","['eq', 'su', 'px', 'eq', 'su', 'su', 'eq', 'px']",eq,12.239845,0.598025,0.371480,2025-12-01 00:22:10+00:00
4,23,2025-12-01 00:24:50.006000+00:00,4,"['MILD', 'RCS', 'RER', 'STAR']","['su', 'su', 'px', 'su']",su,5.882515,0.678975,0.382145,2025-12-01 00:24:50.006000+00:00
...,...,...,...,...,...,...,...,...,...,...
3995,23840,2025-12-17 13:18:00.002857+00:00,4,"['BHW', 'GPW', 'LON', 'RCM']","['su', 'su', 'eq', 'su']",su,8.066754,0.884984,0.538197,2025-12-17 13:18:00.002857+00:00
3996,23848,2025-12-17 13:26:31.240000+00:00,7,"['BHW', 'DART', 'HOPR', 'RCM', 'RER', 'STAR', ...","['px', 'su', 'eq', 'eq', 'su', 'eq', 'eq']",eq,12.225237,0.802749,0.435631,2025-12-17 13:26:31.240000+00:00
3997,23855,2025-12-17 13:36:36.680000+00:00,4,"['HOPR', 'RCM', 'STAR', 'WRW']","['eq', 'eq', 'su', 'su']",eq,10.996588,0.676555,0.419558,2025-12-17 13:36:36.680000+00:00
3998,23860,2025-12-17 13:41:50+00:00,4,"['BHW', 'LO2', 'SHUK', 'WRW']","['eq', 'su', 'su', 'su']",su,30.337232,0.768701,0.493594,2025-12-17 13:41:50+00:00


In [50]:
# Just compare dates, not times - this ignores timezone entirely
december_events_ml = common_events[
    (common_events['to_timestamp'].dt.date >= datetime(2025, 12, 5).date()) & 
    (common_events['to_timestamp'].dt.date <= datetime(2025, 12, 18).date())
]

print(f"Found {len(december_events_ml)} events")
print(december_events_ml)

Found 2701 events
      cluster_id                     rounded_start  num_stations  \
1299        5584  2025-12-05 00:01:08.500000+00:00             5   
1300        5586         2025-12-05 00:03:00+00:00             4   
1301        5590         2025-12-05 00:06:50+00:00             6   
1302        5593  2025-12-05 00:11:36.800000+00:00             6   
1303        5594         2025-12-05 00:14:10+00:00             5   
...          ...                               ...           ...   
3995       23840  2025-12-17 13:18:00.002857+00:00             4   
3996       23848  2025-12-17 13:26:31.240000+00:00             7   
3997       23855  2025-12-17 13:36:36.680000+00:00             4   
3998       23860         2025-12-17 13:41:50+00:00             4   
3999       23887         2025-12-17 14:08:40+00:00             5   

                                               stations  \
1299            ['CRYS', 'DART', 'GPW', 'GRWR', 'KAVK']   
1300                    ['GNOB', 'GSM', 'RCM', 

In [51]:
december_events_pnsn

,evid,etype,selectflag,to_timestamp,lat,lon,totalarr
10,62210017,su,1,2025-12-17 00:45:41.469999+00:00,48.756167,-121.826000,10
11,62209537,su,1,2025-12-13 12:54:49.519998+00:00,48.774500,-121.853167,7
12,62217776,su,1,2025-12-10 19:57:31.620000+00:00,48.757667,-121.847167,7
13,62216746,su,1,2025-12-05 02:03:34.700000+00:00,46.195000,-122.242500,6


In [55]:
december_events_ml

,cluster_id,rounded_start,num_stations,stations,all_classes,most_common_class,mean_auc,mean_max,mean_prob,to_timestamp
1299,5584,2025-12-05 00:01:08.500000+00:00,5,"['CRYS', 'DART', 'GPW', 'GRWR', 'KAVK']","['eq', 'eq', 'eq', 'su', 'su']",eq,9.365093,0.719878,0.419932,2025-12-05 00:01:08.500000+00:00
1300,5586,2025-12-05 00:03:00+00:00,4,"['GNOB', 'GSM', 'RCM', 'TBLMT']","['su', 'eq', 'eq', 'su']",eq,4.497869,0.701053,0.407718,2025-12-05 00:03:00+00:00
1301,5590,2025-12-05 00:06:50+00:00,6,"['COPP', 'GPW', 'MILD', 'OBSR', 'PANH', 'PARA']","['su', 'su', 'su', 'su', 'su', 'eq']",su,5.717931,0.803198,0.494015,2025-12-05 00:06:50+00:00
1302,5593,2025-12-05 00:11:36.800000+00:00,6,"['GNOB', 'KAVK', 'LONR', 'RCM', 'STYX', 'WRW']","['su', 'su', 'eq', 'su', 'eq', 'eq']",eq,4.593908,0.602696,0.375889,2025-12-05 00:11:36.800000+00:00
1303,5594,2025-12-05 00:14:10+00:00,5,"['BHW', 'COWS', 'GTWY', 'LTY', 'SHUK']","['su', 'eq', 'px', 'su', 'eq']",eq,4.616302,0.631040,0.400851,2025-12-05 00:14:10+00:00
...,...,...,...,...,...,...,...,...,...,...
3995,23840,2025-12-17 13:18:00.002857+00:00,4,"['BHW', 'GPW', 'LON', 'RCM']","['su', 'su', 'eq', 'su']",su,8.066754,0.884984,0.538197,2025-12-17 13:18:00.002857+00:00
3996,23848,2025-12-17 13:26:31.240000+00:00,7,"['BHW', 'DART', 'HOPR', 'RCM', 'RER', 'STAR', ...","['px', 'su', 'eq', 'eq', 'su', 'eq', 'eq']",eq,12.225237,0.802749,0.435631,2025-12-17 13:26:31.240000+00:00
3997,23855,2025-12-17 13:36:36.680000+00:00,4,"['HOPR', 'RCM', 'STAR', 'WRW']","['eq', 'eq', 'su', 'su']",eq,10.996588,0.676555,0.419558,2025-12-17 13:36:36.680000+00:00
3998,23860,2025-12-17 13:41:50+00:00,4,"['BHW', 'LO2', 'SHUK', 'WRW']","['eq', 'su', 'su', 'su']",su,30.337232,0.768701,0.493594,2025-12-17 13:41:50+00:00


In [62]:
from datetime import timedelta

# Create a list to store matches
matches = []

# For each event in december_events_pnsn
for idx, pnsn_row in december_events_pnsn.iterrows():
    pnsn_time = pnsn_row['to_timestamp']
    
    # Define the time window (30 seconds before and after)
    time_lower = pnsn_time - timedelta(seconds= 600)
    time_upper = pnsn_time + timedelta(seconds= 600)
    
    # Find all ML events within this window
    ml_matches = december_events_ml[
        (december_events_ml['to_timestamp'] >= time_lower) & 
        (december_events_ml['to_timestamp'] <= time_upper)
    ]
    
    
    """
    # Store the results
    for ml_idx, ml_row in ml_matches.iterrows():
        time_diff = (ml_row['to_timestamp'] - pnsn_time).total_seconds()
        matches.append({
            'pnsn_evid': pnsn_row['evid'],
            'pnsn_timestamp': pnsn_time,
            'ml_evid': ml_row['evid'],
            'ml_timestamp': ml_row['to_timestamp'],
            'time_diff_seconds': time_diff
        })
        
    """

# Convert to DataFrame
matches_df = pd.DataFrame(matches)

print(f"Found {len(matches_df)} matches")
print(matches_df)

Found 0 matches
Empty DataFrame
Columns: []
Index: []


In [63]:
ml_matches


,cluster_id,rounded_start,num_stations,stations,all_classes,most_common_class,mean_auc,mean_max,mean_prob,to_timestamp
1314,5698,2025-12-05 02:02:18.500000+00:00,11,"['COPP', 'KAVK', 'LO2', 'MBW2', 'MILD', 'OBSR'...","['eq', 'eq', 'su', 'su', 'su', 'su', 'su', 'eq...",su,8.335058,0.753826,0.426933,2025-12-05 02:02:18.500000+00:00
1315,5703,2025-12-05 02:08:00.002857+00:00,5,"['LTY', 'MILD', 'OBSR', 'PANH', 'RCM']","['su', 'su', 'su', 'su', 'su']",su,5.914415,0.724338,0.448818,2025-12-05 02:08:00.002857+00:00
